# Bundle Authoring Guide Notebook

Mirrors the [Bundle Authoring Guide](https://gongjr0.github.io/SymbolicDSGE/latest/guides/bundle_authoring_guide/). Each section follows the same headings as the guide.

Run from `docs/assets/` so the relative path to `MODELS/POST82.yaml` resolves.

In [1]:
import io

import numpy as np
import pandas as pd
from numpy import array, float64

from SymbolicDSGE import BundleBuilder, DSGESolver, Estimator, ModelParser, Shock
from SymbolicDSGE.bayesian import make_prior
from SymbolicDSGE.bundle import SimSpec

from SymbolicDSGE.monte_carlo import MCPipeline
from SymbolicDSGE.monte_carlo.operations import (
    core as c,
    tests as t,
)

## Solve a reference model

In [4]:
# Create the Bundle
bundle = BundleBuilder(created_by="experiment-1")

parser = ModelParser("../../MODELS/POST82.yaml")
model, kalman = parser.get_all()

solver = DSGESolver(model, kalman)
compiled = solver.compile(linearize=False)
sol = solver.solve(
    compiled,
    ss_seed=[0.0, 0.0, 0.0, 0.0, 0.0],
)

# Add the model to the bundle
bundle.add_model(
    "reference",
    model.source_yaml,
    compile_kwargs={"linearize": False},
    solve_kwargs={"ss_seed": [0.0, 0.0, 0.0, 0.0, 0.0]},
)

# The `dgp` passed to `mc_pipeline.run(...)` is a runtime-only argument; it is
# NOT persisted by `add_mc`. To make `loaded.dgp` resolve, the DGP model must be
# bundled explicitly under role "dgp". Here the experiment uses the same model as
# its DGP, so we ship the same YAML (in a misspecification study this would be a
# different model's source).
bundle.add_model(
    "dgp",
    model.source_yaml,
    compile_kwargs={"linearize": False},
    solve_kwargs={"ss_seed": [0.0, 0.0, 0.0, 0.0, 0.0]},
)

## Specify the estimation tab

This section defines a small MCMC run against synthetic observed data, then stores the live `Estimator` and `MCMCResult` in the bundle.

In [5]:
priors = {
    "psi_pi": make_prior(
        distribution="normal",
        parameters={"mean": 1.5, "std": 0.25},
        transform="identity",
    ),
    "psi_x": make_prior(
        distribution="normal",
        parameters={"mean": 0.5, "std": 0.2},
        transform="identity",
    ),
}

rng = np.random.default_rng(0)
observed = rng.standard_normal((40, 3))
estim = Estimator(
    solver=solver,
    compiled=compiled,
    observables=["OutGap", "Infl", "Rate"],
    y=observed,
    priors=priors,
)
res = estim.mcmc(
    n_draws=1000,
    burn_in=200,
    thin=2,
)

MCMC sampling concluded in 1.04 seconds with 2119.41 iterations per second.
[Estimator:mcmc] BK stability warnings encountered during search: 0


In [6]:
# Add the estimation to the bundle with results
bundle.add_estimation(
    source=estim,
    result=res,
)

## Build a Monte Carlo pipeline

In [7]:
# Pass the `Shock` *instances* (not `.shock_generator()`): the pipeline clones
# them per replication with seeded offsets, and they serialize into the bundle.
gz_shock = Shock(seed=0, multivar=True, dist="norm")
r_shock = Shock(seed=1, multivar=False, dist="t", dist_kwargs={"df": 3})

mc_pipeline = MCPipeline(
    [
        c.simulation_step(
            target="dgp",
            T=200,
            shocks={"g,z": gz_shock, "r": r_shock},
        ),
        t.jarque_bera_test_step(
            "jb_test", source="datagen", field="observables", column=0
        ),
    ]
)
mc_res = mc_pipeline.run(
    reference=sol,
    dgp=sol,
    n_rep=1000,
    retain_payloads=False,
    retain_contexts=True,
    verbosity=2,
)

MC run concluded successfully in 0.30s with 3359.33 it/s.
Per-step Report:

	datagen: 0 faliures, 3718.64 it/s (0.27s).
	jb_test: 0 faliures, 40476.00 it/s (0.02s).


In [8]:
# Add the Monte Carlo pipeline to the bundle
bundle.add_mc(
    pipeline=mc_pipeline,
    result=mc_res,
)

C:\Users\guney\Documents\GitHub\SymbolicDSGE\SymbolicDSGE\bundle\builder.py:279: UserWarning: MC results were ran with retain_test_results=True, retain_contexts=True. Payloads, per-rep TestResult objects, and per-rep MCContext objects are not supported in the bundle and will be missing.
  w.warn(


## Specify a simulation prefill

In [9]:
simulation = SimSpec(
    T=200,
    observables=True,
    shock_scale=1.0,
    shocks={
        "r": Shock(seed=42, dist="norm", dist_kwargs={"loc": 0.0}).to_dict(),
    },
)

# Add the simulation prefill to the bundle, keyed by role
bundle.set_simulation("reference", simulation)

## Add raw data

In [10]:
# Make text data
aux = pd.DataFrame(
    {
        "date": pd.date_range("2000-01-01", periods=40, freq="QS"),
        "gdp_growth": rng.standard_normal(40),
    }
)
csv_buf = io.StringIO()
aux.to_csv(csv_buf, index=False)

bundle.add_raw_data(
    name="auxiliary_series",
    data=csv_buf.getvalue(),
)

## Write the bundle to disk

In [11]:
bundle.write("experiment-1.sdsge")

WindowsPath('experiment-1.sdsge')

## Inspect the result

In [12]:
!unzip -l experiment-1.sdsge

Archive:  experiment-1.sdsge
  Length      Date    Time    Name
---------  ---------- -----   ----
     3204  22-07-2026 00:19   manifest.json
     2454  22-07-2026 00:19   model/reference.yaml
     2454  22-07-2026 00:19   model/dgp.yaml
      936  22-07-2026 00:19   estimation/spec.json
      382  22-07-2026 00:19   estimation/result.json
     1939  22-07-2026 00:19   estimation/observed.parquet
    18602  22-07-2026 00:19   estimation/posterior.parquet
     1046  22-07-2026 00:19   montecarlo/pipeline.json
     4575  22-07-2026 00:19   montecarlo/result.json
    13882  22-07-2026 00:19   montecarlo/traces.parquet
     1282  22-07-2026 00:19   data/auxiliary_series.parquet
---------                     -------
    50756                     11 files


In [11]:
# File size of the bundle in KB
import os

sizeof = round(os.path.getsize("experiment-1.sdsge") / 1024)
print(f"Bundle size: {sizeof} KB")

Bundle size: 42 KB
